<a href="https://colab.research.google.com/github/himanshubhimte69/AgriLite-A-Lightweight-Multi-Crop-Disease-Detector-Across-Diverse-Conditions/blob/main/Embrapa_DenseNet121_FineTune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"himanshubhimte","key":"6ee92afc21406e7e1c24e75894c8d5f5"}'}

In [ ]:
!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

#Embrapa dataset
!kaggle datasets download -d poornimasinghthakur/embrapa
!unzip embrapa.zip

Streaming output truncated to the last 5000 lines.
  inflating: XDB/val/Feijao (Dry bean) - Antracnose (Anthracnose) - Cropped/antr137.jpg  
  inflating: XDB/val/Feijao (Dry bean) - Antracnose (Anthracnose) - Cropped/antr145.jpg  
  inflating: XDB/val/Feijao (Dry bean) - Antracnose (Anthracnose) - Cropped/antr155.jpg  
  inflating: XDB/val/Feijao (Dry bean) - Antracnose (Anthracnose) - Cropped/antr165.jpg  
  inflating: XDB/val/Feijao (Dry bean) - Antracnose (Anthracnose) - Cropped/antr177.jpg  
  inflating: XDB/val/Feijao (Dry bean) - Antracnose (Anthracnose) - Cropped/antr193.jpg  
  inflating: XDB/val/Feijao (Dry bean) - Antracnose (Anthracnose) - Cropped/antr195.jpg  
  inflating: XDB/val/Feijao (Dry bean) - Antracnose (Anthracnose) - Cropped/antr196.jpg  
  inflating: XDB/val/Feijao (Dry bean) - Antracnose (Anthracnose) - Cropped/antr208.jpg  
  inflating: XDB/val/Feijao (Dry bean) - Antracnose (Anthracnose) - Cropped/antr217.jpg  
  inflating: XDB/val/Feijao (Dry bean) - Antracno

In [ ]:
import tensorflow as tf

img_size = (224, 224)  # For DenseNet121
batch_size = 16

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "/content/XDB/train",
    image_size=img_size,
    batch_size=batch_size
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "/content/XDB/val",
    image_size=img_size,
    batch_size=batch_size
)

test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "/content/XDB/test",
    image_size=img_size,
    batch_size=batch_size
)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

Found 29607 files belonging to 93 classes.
Found 7455 files belonging to 93 classes.
Found 9314 files belonging to 93 classes.
Classes: ['AlgodтФЬ├║o (Cotton) - Mancha de Mirotecio (Myrothecium leaf spot) - Cropped', 'AlgodтФЬ├║o (Cotton) - Mela (Soreshin) - Cropped', 'AlgodтФЬ├║o (Cotton) - Ramularia (Areolate Mildew) - Cropped', 'Arroz (Rice) - Brusone (Rice Blast) - Cropped', 'Arroz (Rice) - Escaldadura (Leaf Scald) - Cropped', 'CafтФЬтМР (Coffee) - BichoMineiro (Leaf Miner) - Cropped', 'CafтФЬтМР (Coffee) - Cercosporiose (Cercospora Leaf Spot) - Cropped', 'CafтФЬтМР (Coffee) - Ferrugem (Rust) - Cropped', 'CafтФЬтМР (Coffee) - Mancha Aureolada (Bacterial Blight) - Cropped', 'CafтФЬтМР (Coffee) - Mancha Mantegosa (Blister Spot) - Cropped', 'CafтФЬтМР (Coffee) - Phoma (Brown leaf spot) - Cropped', 'Cajueiro (Cashew Tree) - Alga (Algae) - Cropped', 'Cajueiro (Cashew Tree) - Antracnose (Anthracnose) - Cropped', 'Cajueiro (Cashew Tree) - Mancha Angular (Angular Leaf Spot) - Cropped', 'Ca

In [ ]:
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras import layers, models

base_model = DenseNet121(
    include_top=False,
    weights='imagenet',
    input_shape=(224, 224, 3)
)
base_model.trainable = False  # Freeze the pretrained base

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       262,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 93)             │        23,901 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,323,805 (27.94 MB)

 Trainable params: 286,301 (1.09 MB)

 Non-trainable params: 7,037,504 (26.85 MB)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
1851/1851 ━━━━━━━━━━━━━━━━━━━━ 171s 75ms/step - accuracy: 0.2577 - loss: 3.4673 - val_accuracy: 0.4991 - val_loss: 1.8382
Epoch 2/10
1851/1851 ━━━━━━━━━━━━━━━━━━━━ 101s 55ms/step - accuracy: 0.3855 - loss: 2.2536 - val_accuracy: 0.5569 - val_loss: 1.6019
Epoch 3/10
1851/1851 ━━━━━━━━━━━━━━━━━━━━ 141s 54ms/step - accuracy: 0.4312 - loss: 2.0328 - val_accuracy: 0.5830 - val_loss: 1.4877
Epoch 4/10
1851/1851 ━━━━━━━━━━━━━━━━━━━━ 100s 54ms/step - accuracy: 0.4443 - loss: 1.9568 - val_accuracy: 0.5869 - val_loss: 1.4224
Epoch 5/10
1851/1851 ━━━━━━━━━━━━━━━━━━━━ 100s 54ms/step - accuracy: 0.4627 - loss: 1.8923 - val_accuracy: 0.5969 - val_loss: 1.3757
Epoch 6/10
1851/1851 ━━━━━━━━━━━━━━━━━━━━ 100s 54ms/step - accuracy: 0.4678 - loss: 1.8732 - val_accuracy: 0.6140 - val_loss: 1.3308
Epoch 7/10
1851/1851 ━━━━━━━━━━━━━━━━━━━━ 100s 54ms/step - accuracy: 0.4755 - loss: 1.8316 - val_accuracy: 0.6039 - val_loss: 1.4037
Epoch 8/10
1851/1851 ━━━━━━━━━━━━━━━━━━━━ 142s 54ms/step - accuracy: 

In [ ]:
# ---- Fine-Tune DenseNet121 ----

base_model.trainable = True   # unfreeze the full model

# Freeze first 250 layers (best for DenseNet121)
for layer in base_model.layers[:250]:
    layer.trainable = False

from tensorflow.keras.optimizers import Adam

# Recompile with low LR
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train again
history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
1851/1851 ━━━━━━━━━━━━━━━━━━━━ 298s 118ms/step - accuracy: 0.1413 - loss: 7.8558 - val_accuracy: 0.4663 - val_loss: 1.9553
Epoch 2/10
1851/1851 ━━━━━━━━━━━━━━━━━━━━ 146s 79ms/step - accuracy: 0.3679 - loss: 2.3606 - val_accuracy: 0.5586 - val_loss: 1.4702
Epoch 3/10
1851/1851 ━━━━━━━━━━━━━━━━━━━━ 146s 79ms/step - accuracy: 0.4576 - loss: 1.9250 - val_accuracy: 0.6304 - val_loss: 1.2641
Epoch 4/10
1851/1851 ━━━━━━━━━━━━━━━━━━━━ 144s 78ms/step - accuracy: 0.5029 - loss: 1.7320 - val_accuracy: 0.6702 - val_loss: 1.1351
Epoch 5/10
1851/1851 ━━━━━━━━━━━━━━━━━━━━ 145s 78ms/step - accuracy: 0.5353 - loss: 1.5948 - val_accuracy: 0.6963 - val_loss: 1.0529
Epoch 6/10
1851/1851 ━━━━━━━━━━━━━━━━━━━━ 144s 78ms/step - accuracy: 0.5635 - loss: 1.4793 - val_accuracy: 0.7187 - val_loss: 0.9893
Epoch 7/10
1851/1851 ━━━━━━━━━━━━━━━━━━━━ 144s 78ms/step - accuracy: 0.5941 - loss: 1.3951 - val_accuracy: 0.7321 - val_loss: 0.9444
Epoch 8/10
1851/1851 ━━━━━━━━━━━━━━━━━━━━ 203s 78ms/step - accuracy:

In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, cohen_kappa_score, classification_report
)
from sklearn.preprocessing import label_binarize

# Get true labels from test_ds
y_true = []
y_pred = []

for images, labels in test_ds:
    probs = model.predict(images, verbose=0)
    preds = np.argmax(probs, axis=1)
    y_true.extend(labels.numpy())
    y_pred.extend(preds)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')
kappa = cohen_kappa_score(y_true, y_pred)

# AUC requires binarized labels and predicted probabilities
# Get full probability predictions
y_probs = []

for images, _ in test_ds:
    y_probs.extend(model.predict(images, verbose=0))

y_probs = np.array(y_probs)
y_true_bin = label_binarize(y_true, classes=range(num_classes))

auc = roc_auc_score(y_true_bin, y_probs, average='macro', multi_class='ovr')

# ✅ Print all metrics
print("\n Evaluation Metrics:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"AUC      : {auc:.4f}")
print(f"Kappa    : {kappa:.4f}")

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



 Evaluation Metrics:
Accuracy : 0.7701
Precision: 0.7527
Recall   : 0.7701
F1 Score : 0.7441
AUC      : 0.5030
Kappa    : 0.7601
